# Phase 7: Walk-forward backtest

Each test season (2019–2025) is predicted by a model trained only on earlier seasons, with shrinkage, scaling, and the logistic regression all refit inside every fold. All the logic is in `src/nfl_ats/`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

from nfl_ats.data import HISTORY_SEASONS, PRIOR_SEASON, load_schedules, regular_season
from nfl_ats.evaluate import (BREAKEVEN, TEST_SEASONS, brier_decomposition, reliability_table, score,
                              season_table)
from nfl_ats.features import FEATURES, load_team_games
from nfl_ats.model import walk_forward
from nfl_ats.target import add_home_cover

sns.set_theme(style="whitegrid")
FIG_DIR = "../reports/figures"

In [ ]:
games = add_home_cover(regular_season(load_schedules()))
team_games = load_team_games([PRIOR_SEASON] + HISTORY_SEASONS)
preds = walk_forward(games, team_games, FEATURES, TEST_SEASONS)
preds.shape

## 1. Report card: model vs. baselines, per season and pooled

In [ ]:
table = season_table(preds, games)
table.round(3)

In [ ]:
by_season = table.drop(index="pooled").reset_index()
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.errorbar(by_season["season"], by_season["accuracy"],
            yerr=[by_season["accuracy"] - by_season["ci_low"], by_season["ci_high"] - by_season["accuracy"]],
            fmt="o", capsize=5, color="steelblue", lw=2, label="v1 model (95% CI)")
for name, marker in [("always_home", "s"), ("always_favorite", "^"), ("coin_flip", "x")]:
    ax.scatter(by_season["season"], by_season[name], marker=marker, color="gray", alpha=0.7,
               label=name.replace("_", " "))
pooled = table.loc["pooled"]
ax.axhspan(pooled["ci_low"], pooled["ci_high"], color="steelblue", alpha=0.12,
           label=f"pooled {pooled['accuracy']:.1%} [{pooled['ci_low']:.1%}, {pooled['ci_high']:.1%}]")
ax.axhline(BREAKEVEN, color="crimson", ls=":", lw=1.5, label="52.4% breakeven")
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set(ylim=(0.38, 0.62), xlabel="Test season", ylabel="ATS accuracy (pushes excluded)",
       title="v1 walk-forward: every season's CI includes 50%, and none clears breakeven")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=8, loc="lower right", ncol=2)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/v1_backtest_by_season.png", dpi=150)

In [ ]:
# Which side does the model pick? With weak features, the training base rate drives it.
base_rate = {s: games.query("2016 <= season < @s")["home_cover"].mean() for s in TEST_SEASONS}
(preds.groupby("season")
      .agg(home_pick_share=("p", lambda p: (p > 0.5).mean()), mean_p=("p", "mean"), p_sd=("p", "std"))
      .assign(train_home_cover_rate=pd.Series(base_rate))
      .round(3))

**Takeaway:** pooled over 1,828 games, v1 hits **51.3% [48.9, 53.5]**, so no edge. Log loss is 0.695 and Brier 0.251, both a bit *worse* than a constant 0.5 (0.693, 0.250). Season accuracy ranges from 49.6% to 52.7%, and every CI includes 50%.

**Why it mostly picks away:** each fold's average probability equals its training home-cover rate. Once 2019 (43% home covers) entered training, that rate fell from 49.9% to about 48.3%. The features move probabilities by only about ±2 points (SD 0.016–0.045, *shrinking* as training seasons are added, so more data teaches the model the clues matter even less). That small drop pushed most games below 50% and the model picked away 68–82% of the time in 2020–2024. That's a sign the features carry almost no signal: the intercept, not the clues, decides most picks.

## 2. Calibration: when the model says X%, does the home team cover X% of the time?

Games are split into 10 equal-count bins by predicted probability. A calibrated model's points sit on the diagonal. A model with *skill* also spreads its points out along it.

In [ ]:
bins = reliability_table(preds["home_cover"], preds["p"]).reset_index(drop=True)
bins["se"] = np.sqrt(bins["cover_rate"] * (1 - bins["cover_rate"]) / bins["games"])

fig, (ax, hx) = plt.subplots(2, 1, figsize=(7, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(bins["mean_p"], bins["cover_rate"], yerr=1.96 * bins["se"], fmt="o", capsize=4, color="steelblue",
            label="observed cover rate (95% CI), about 183 games per bin")
ax.plot([0.3, 0.7], [0.3, 0.7], color="gray", ls="--", lw=1, label="perfect calibration")
ax.axhline(preds["home_cover"].mean(), color="crimson", ls=":", lw=1, label="overall cover rate")
ax.set(xlim=(0.3, 0.7), ylim=(0.3, 0.7), ylabel="Actual home cover rate",
       title="Reliability: predictions cluster near 48%, and outcomes don't follow them")
ax.legend(fontsize=8, loc="upper left")
sns.histplot(preds["p"], binwidth=0.005, ax=hx, color="steelblue")
hx.set(xlabel="Predicted P(home covers)", ylabel="Games")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/v1_reliability.png", dpi=150)

In [ ]:
# Brier = reliability - resolution + uncertainty. Resolution has a noise floor: with about 183 games
# per bin, random outcomes alone produce resolution of about uncertainty / games_per_bin.
decomp = brier_decomposition(preds["home_cover"], preds["p"])
decomp["resolution_noise_floor"] = decomp["uncertainty"] / (len(preds) / 10)
decomp.round(5)

**Takeaway:** the model is *roughly* calibrated in that its predictions (44–54% in nearly every bin) sit near the real cover rate. But it has **no resolution**. Measured resolution (0.0013) equals the noise floor you'd get from random outcomes (about 0.0014), and the bins don't trend upward: the lowest bin covered 53%, and the highest 50%. Reliability error (0.0018) is larger than resolution, which is why Brier (0.2508) ends up slightly *worse* than the constant-0.5 forecast (0.2500). In short, the probabilities are honest about being unsure, and they carry no information about who covers.